# Guide 5 — Reading One Card (grid positioning)

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

Now we put Guides 2–4 together. When the referee says "a card was flipped at `C4`,"
the program needs to *read that one card*. It lines up the board with the corner
markers, cuts out just square `C4`, and asks YOLO what it sees.

If the first look isn't clear, it tries again while being a little less picky
(lower confidence), and takes a couple of photos to be safe. All real code.


### How this guide fits in

**Depends on:** Guides 2, 3, and 4 (it uses the camera, the grid math, and the corner markers). **Used by:** the main turn loop (Guide 0).

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### The main function: `detect_position`

Give it a square like `C4` and it returns what's on the card and how sure it is.
This runs automatically during a match — you never call it by hand. Read the
comments to see the "try again, a little less picky each time" loop.


In [ ]:
def detect_position(pos, show=True):
    """Auto-calibrate from border markers, crop only `pos`, and run YOLO on
    that perspective-corrected 416x416 cell. Retries fresh photos at lower
    confidence thresholds without ever falling back to full-frame inference."""
    row, col = parse_pos(pos)
    if not (0 <= row < GRID_ROWS and 0 <= col < GRID_COLS):
        raise ValueError(f'Position outside configured grid: {pos!r}')

    for threshold in turn_score_thresholds():
        for photo in range(DETECT_PHOTO_COUNT):
            frame = capture_frame(
                flush_frames=DETECT_FLUSH_FRAMES if photo == 0 else 0,
                settle_seconds=DETECT_SETTLE_SECONDS if photo == 0 else 0.15,
            )
            marker_count = calibrate_from_border_markers(frame)
            if marker_count < 4 and BOARD_CORNERS is None:
                print(f'[aruco_border] {marker_count}/4 border markers visible -- cannot crop {pos} yet')
                continue
            record = best_detection_for_grid_cell(frame, row, col, score_thresh=threshold)
            if record is not None:
                if show:
                    show_detection_frame(frame, [record], turn_label=f'{pos} grid-cell crop')
                return record['description'], record.get('score', 1.0)

    print(f'[detect] WARNING: no object detected in the {pos} grid-cell crop even at the lowest threshold.')
    if show:
        frame = capture_frame(flush_frames=0, settle_seconds=0.0)
        show_detection_frame(frame, [], turn_label=f'{pos} grid-cell crop')
    return 'unknown', 0.0


def detect_position_debug(pos):
    """Manual, one-off test of detection at a position. Does not touch
    match state or send anything to the referee -- setup/debug only."""
    cls, score = detect_position(pos, show=True)
    print(f'{pos}: {cls} (score={score:.2f})')
    return cls, score

### A helper for testing

`camera_debug_one_shot` lets you test your camera and one square *before* a real
match, so you can check that detection works. Great for practice.


In [ ]:
def camera_debug_one_shot(pos='A1'):
    """Auto-orient the board from corner-marker roles and run YOLO on one cell.

    The left dashboard image is the canonical full-board view: the marker
    configured as TL is mapped to its top-left and A1 is highlighted there,
    regardless of camera rotation. The right image is the identically oriented
    416x416 cell crop passed to YOLO.
    """
    pos = str(pos).strip().upper()
    row, col = parse_pos(pos)
    if not (0 <= row < GRID_ROWS and 0 <= col < GRID_COLS):
        raise ValueError(f'Position outside configured grid: {pos!r}')

    frame = capture_frame(flush_frames=4, settle_seconds=0.2)
    marker_count = calibrate_from_border_markers(frame)
    if marker_count < 4:
        raise RuntimeError(f'Only {marker_count}/4 border markers visible; cannot orient the board.')

    crop, _, _ = crop_grid_cell(frame, row, col)
    boxes, scores, classes = run(crop)

    board_view = draw_canonical_board(frame, selected_pos=pos)
    _set_image_widget(alignment_image_widget, board_view)
    show_processed_grid_crop(crop, boxes, scores, classes, pos)

    print(f'{pos}: {len(boxes)} object(s) detected in the processed grid-cell crop:')
    for score, class_idx in sorted(zip(scores, classes), reverse=True):
        print(f'  {class_names[int(class_idx)]}: {float(score):.2f}')
    return boxes, scores, classes

### Check yourself

1. Why does `detect_position` take more than one photo?
2. If a card can't be read at all, what does the function return? Why does that
   matter for the matching guide later?
